In [8]:
from  langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages  import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [9]:
load_dotenv()

True

In [10]:
model = ChatGroq(model="qwen/qwen3.8-27b")


In [11]:
from langgraph.graph.message  import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [12]:
def chat_node(state: ChatState):
    #take user query from state
    messages = state['messages']
    # send  to llm
    response = model.invoke(messages)
    # response store state
    return {'messages': [response]}

In [14]:
checkpoint = MemorySaver()

graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

# add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)


chatbot = graph.compile(checkpointer=checkpoint)

In [22]:
thread_id = '1'

initial_state = {'messages': [HumanMessage(content='what is my name again?')]}
config = {
    'configurable': {
        'thread_id': thread_id
        }
    }

response = chatbot.invoke(initial_state, config=config)

print(response)

{'messages': [HumanMessage(content='My name is patrick', additional_kwargs={}, response_metadata={}, id='82a3ae18-cf8c-449b-837d-b1bf01b6b4ea'), AIMessage(content='Nice to meet you, Patrick! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 17, 'total_tokens': 32, 'completion_time': 0.032110029, 'completion_tokens_details': None, 'prompt_time': 0.000874399, 'prompt_tokens_details': None, 'queue_time': 0.04887872, 'total_time': 0.032984428}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1b1e2-7499-77bf-9a28-7b4f3b52693e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 15, 'total_tokens': 32}), HumanMessage(content='My name is patrick', additional_kwargs={}, response_metadata={}, id='fb204772-5e4c-4698-b6d5-74da850fc578'), AIMes